In [ ]:
#Step 1: Import Libraries
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, broadcast

In [ ]:
#Step 2: Create Spark Session
spark = SparkSession.builder\
.appName("Scalability Concepts")\
.getOrCreate()
#Why: Every Spark job needs a session. The app name helps identify jobs in logs/UI.

In [ ]:
#Step 3 & 4: Load Bronze and Silver Activity Datasets
bronze_activity_df = spark.read.option("header", True).option("inferSchema", True).csv(r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\02.medallion_architecture\01.bronze\user_activity.csv")

silver_activity_df = spark.read.option("header", True).option("inferSchema", True).csv(r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\02.medallion_architecture\02.silver\user_activity.csv")

In [10]:
bronze_activity_df.count(), silver_activity_df.count()

(5000, 5000)

In [ ]:
#Schema Validation
bronze_activity_df.printSchema()
silver_activity_df.printSchema()

root
 |-- activity_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- show_id: string (nullable = true)
 |-- watch_minutes: integer (nullable = true)

root
 |-- activity_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- show_id: string (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- watch_hours: double (nullable = true)



In [ ]:
#Data Quality Check: Negative Watch Minutes
silver_activity_df.filter(col("watch_minutes") < 0).count()

0

In [ ]:
#Data Quality Check: Negative Watch Minutes
bronze_activity_df.filter(col("watch_minutes") < 0).count()

0

In [ ]:
#Step 5: Load Silver Users Dataset
users_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\02.medallion_architecture\02.silver\users.csv")

users_df.show(5)
users_df.printSchema()

+-------+---------+---------+-----------------+
|user_id|user_name|  country|subscription_type|
+-------+---------+---------+-----------------+
|      1|   User_1|   Canada|         Standard|
|      2|   User_2|Australia|            Basic|
|      3|   User_3|       UK|          Premium|
|      4|   User_4|   Canada|          Premium|
|      5|   User_5|  Germany|          Premium|
+-------+---------+---------+-----------------+
only showing top 5 rows
root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)



In [ ]:
#Step 7: Lazy Evaluation
# Transformation only
# Spark does not execute this immediately.

filtered_activity_df = bronze_activity_df.filter(
    col("watch_minutes") > 100
)

In [ ]:
#  View logical plan

filtered_activity_df.explain(True)
# Why: Transformations don’t execute until an action is triggered. explain(True) shows the logical plan.

== Parsed Logical Plan ==
'Filter '`>`('watch_minutes, 100)
+- Relation [activity_id#56,user_id#57,show_id#58,watch_minutes#59] csv

== Analyzed Logical Plan ==
activity_id: int, user_id: int, show_id: string, watch_minutes: int
Filter (watch_minutes#59 > 100)
+- Relation [activity_id#56,user_id#57,show_id#58,watch_minutes#59] csv

== Optimized Logical Plan ==
Filter (isnotnull(watch_minutes#59) AND (watch_minutes#59 > 100))
+- Relation [activity_id#56,user_id#57,show_id#58,watch_minutes#59] csv

== Physical Plan ==
*(1) Filter (isnotnull(watch_minutes#59) AND (watch_minutes#59 > 100))
+- FileScan csv [activity_id#56,user_id#57,show_id#58,watch_minutes#59] Batched: false, DataFilters: [isnotnull(watch_minutes#59), (watch_minutes#59 > 100)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/jovit/OneDrive/Documents/GitHub/data-engineering-90day-..., PartitionFilters: [], PushedFilters: [IsNotNull(watch_minutes), GreaterThan(watch_minutes,100)], ReadSchema: struct<activity

In [ ]:
# Step 8: Action
# count() is an action.
# This triggers Spark execution.

filtered_count = filtered_activity_df.count()

filtered_count
#Why: Actions like count() trigger execution. Spark reads data, applies filters, and computes results.

3435

In [ ]:

# Step 9: Transformation vs Action
# Transformations: lazy operations

transformed_df = bronze_activity_df \
    .select("user_id", "show_id", "watch_minutes") \
    .filter(col("watch_minutes") > 100)

In [19]:
# Action: triggers execution

transformed_df.show(5)

+-------+-------+-------------+
|user_id|show_id|watch_minutes|
+-------+-------+-------------+
|    313|  s7076|          134|
|    455|  s6205|          126|
|    205|  s2100|          112|
|    453|  s7494|          147|
|    473|   s776|          223|
+-------+-------+-------------+
only showing top 5 rows


In [20]:
# Check number of partitions in Bronze dataset

bronze_activity_df.rdd.getNumPartitions()

1

In [ ]:
#Step 10: Check Partitions
# Repartition changes the number of partitions.
# This can improve parallelism but causes shuffle.

repartitioned_activity_df = bronze_activity_df.repartition(4)
#Why: Partitions define parallelism. Too few = bottleneck, too many = overhead.

In [ ]:
#Step 11: Repartition Dataset
repartitioned_activity_df.rdd.getNumPartitions()
#Why: Increases partitions for parallelism. Causes shuffle (expensive).

4

In [ ]:
#import sum function for aggregation
from pyspark.sql.functions import sum

In [ ]:
#Step 12: Demonstrate Shuffle
# groupBy causes shuffle because same user_id values
# must be moved to the same partition.

user_watch_summary_df = bronze_activity_df.groupBy(
    "user_id"
).agg(
    sum("watch_minutes").alias("total_watch_minutes")
)

In [ ]:
user_watch_summary_df.explain(True)
#Why: GroupBy requires moving data with same key to same partition → shuffle. Look for Exchange in plan.

== Parsed Logical Plan ==
'Aggregate ['user_id], ['user_id, 'sum('watch_minutes) AS total_watch_minutes#189]
+- Relation [activity_id#56,user_id#57,show_id#58,watch_minutes#59] csv

== Analyzed Logical Plan ==
user_id: int, total_watch_minutes: bigint
Aggregate [user_id#57], [user_id#57, sum(watch_minutes#59) AS total_watch_minutes#189L]
+- Relation [activity_id#56,user_id#57,show_id#58,watch_minutes#59] csv

== Optimized Logical Plan ==
Aggregate [user_id#57], [user_id#57, sum(watch_minutes#59) AS total_watch_minutes#189L]
+- Project [user_id#57, watch_minutes#59]
   +- Relation [activity_id#56,user_id#57,show_id#58,watch_minutes#59] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[user_id#57], functions=[sum(watch_minutes#59)], output=[user_id#57, total_watch_minutes#189L])
   +- Exchange hashpartitioning(user_id#57, 200), ENSURE_REQUIREMENTS, [plan_id=349]
      +- HashAggregate(keys=[user_id#57], functions=[partial_sum(watch_minutes#59)], output=[

In [ ]:
#Step 13: Gold Aggregation
user_summary_df = silver_activity_df.groupBy(
    "user_id"
).agg(
    sum("watch_minutes").alias("total_watch_minutes"),
    sum("watch_hours").alias("total_watch_hours")
)


In [28]:
user_summary_df.show(5)

+-------+-------------------+------------------+
|user_id|total_watch_minutes| total_watch_hours|
+-------+-------------------+------------------+
|    148|               1252|20.880000000000003|
|    471|               1332|              22.2|
|    463|               2190|             36.49|
|    496|               1172|             19.53|
|    243|                885|14.740000000000002|
+-------+-------------------+------------------+
only showing top 5 rows


In [ ]:
#Step 14: Cache Gold Dataset
# Cache stores the DataFrame in memory for reuse.
user_summary_df.cache()

DataFrame[user_id: int, total_watch_minutes: bigint, total_watch_hours: double]

In [30]:
# Action required to materialize cache.

user_summary_df.count()

500

In [ ]:
# Verify cache status.

user_summary_df.is_cached
#Why: Cache avoids recomputation of expensive transformations. Useful for iterative queries.

True

In [ ]:
#Step 15: Broadcast Join
# Broadcast users_df because it is a small lookup/dimension table.
activity_users_df = silver_activity_df.join(
    broadcast(users_df),
    "user_id",
    "inner"
)
#Why: Broadcast small tables to all nodes → avoids shuffle → faster joins.

In [ ]:
#Step 16: Verify Broadcast Join
activity_users_df.show(5)


+-------+-----------+-------+-------------+-----------+---------+-------+-----------------+
|user_id|activity_id|show_id|watch_minutes|watch_hours|user_name|country|subscription_type|
+-------+-----------+-------+-------------+-----------+---------+-------+-----------------+
|    313|          1|  s7076|          134|       2.23| User_313| France|          Premium|
|    455|          2|  s6205|          126|        2.1| User_455| Canada|            Basic|
|    317|          3|  s2292|           10|       0.17| User_317| Canada|          Premium|
|    205|          4|  s2100|          112|       1.87| User_205|  India|         Standard|
|    453|          5|  s7494|          147|       2.45| User_453|Germany|         Standard|
+-------+-----------+-------+-------------+-----------+---------+-------+-----------------+
only showing top 5 rows


In [ ]:
activity_users_df.explain(True)
#Why: Look for BroadcastHashJoin in plan → confirms optimization applied.

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [user_id])
:- Relation [activity_id#77,user_id#78,show_id#79,watch_minutes#80,watch_hours#81] csv
+- ResolvedHint (strategy=broadcast)
   +- Relation [user_id#137,user_name#138,country#139,subscription_type#140] csv

== Analyzed Logical Plan ==
user_id: int, activity_id: int, show_id: string, watch_minutes: int, watch_hours: double, user_name: string, country: string, subscription_type: string
Project [user_id#78, activity_id#77, show_id#79, watch_minutes#80, watch_hours#81, user_name#138, country#139, subscription_type#140]
+- Join Inner, (user_id#78 = user_id#137)
   :- Relation [activity_id#77,user_id#78,show_id#79,watch_minutes#80,watch_hours#81] csv
   +- ResolvedHint (strategy=broadcast)
      +- Relation [user_id#137,user_name#138,country#139,subscription_type#140] csv

== Optimized Logical Plan ==
Project [user_id#78, activity_id#77, show_id#79, watch_minutes#80, watch_hours#81, user_name#138, country#139, subscription_type#140]
